# Data-quality masking

This example uses five simulated recordings with movement-related disturbances. You will see how to flag questionable portions of the signal, review those flags, and carry them through common analysis steps so that poor-quality measurements are not mistaken for meaningful brain activity.

## Create a simulated dataset

The data are simulated so the example can be repeated without a subject recording. The movement disturbances are included to make it easier to see why quality checks matter.

In [1]:
from pyBrainAnalyzIR.testing.simData import simMotionArtifact, simdataset

dset, _truth = simdataset(
    num_files=5,
    snr=5,
    modifiers=[simMotionArtifact],
    random_seed=2026,
)
print(f"Created {len(dset.dataset)} recordings")

 94%|█████████▍| 15/16 [00:00<00:00, 3394.37it/s]

Created 5 recordings


## Flag movement-related disturbances

This step searches for abrupt changes that can be caused by movement. It adds a quality flag to each recording; a flagged point should be treated cautiously, rather than interpreted as a reliable measurement.

In [2]:
import pyBrainAnalyzIR.pipelines.modules as pipeline_modules

motion_mask = pipeline_modules.data_masking.MotionArtifactDetectionMask()
motion_mask.show()
dset = motion_mask.run(dset)

[(rec.masks["motion_artifact_detection"].dims,
  rec.masks["motion_artifact_detection"].dtype)
 for rec in dset.dataset]

Motion artifact detection mask:
Options:
	t_motion     : 0.5
	t_mask       : 1.0
	stdev_thresh : 50.0
	amp_thresh   : 5.0


[(('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool'))]

## See how the quality flags affect analysis

These examples use copies of the same recording for a fair comparison. When the data are resampled, flagged portions remain identified. The correction step uses the quality flags while reducing movement-related variation, and the statistical model excludes flagged measurements when estimating the response.

In [3]:
import copy
import numpy as np
import pyBrainAnalyzIR.pipelines.modules.preproccessing as prep
import pyBrainAnalyzIR.pipelines.modules.glm as glm_module

recording = copy.deepcopy(dset.dataset[0])
motion_mask = recording.masks["motion_artifact_detection"]
bad_input_samples = int((~motion_mask).sum().item())
print(f"Detected {bad_input_samples} bad time/channel samples in this recording")
assert bad_input_samples > 0, "The simulated motion artifacts should produce a non-empty mask."

# Resampling preserves the mask and puts NaNs back at the projected bad samples.
resampled_recording = copy.deepcopy(recording)
resample_job = pipeline_modules.preproccessing.resample()
resample_job.options["Fs"] = 2
resampled_recording = resample_job.run(resampled_recording)
resampled_mask = resampled_recording.masks["motion_artifact_detection"]
resampled_bad_samples = int((~resampled_mask).sum().item())
resampled_nan_samples = int(
    np.isnan(resampled_recording["amp"].pint.dequantify().to_numpy())
    .any(axis=-1).sum()
)
print(
    f"Resampled recording: {resampled_bad_samples} masked samples, "
    f"{resampled_nan_samples} samples restored as NaN"
)
assert resampled_bad_samples > 0 and resampled_nan_samples > 0

# Recursive PCA-based motion correction consumes the recording's quality mask.
pca_recording = copy.deepcopy(recording)
pca_job = pipeline_modules.motion_correction.MotionCorrectPCA()
pca_job.options["recursive"] = True
pca_recording = pca_job.run(pca_recording)
pca_bad_samples = int(
    (~pca_recording.masks["motion_artifact_detection"]).sum().item()
)
print(f"PCA-corrected recording retains {pca_bad_samples} masked samples")
assert pca_bad_samples == bad_input_samples

# The GLM module combines the recording masks and omits bad samples per channel.
glm_recording = copy.deepcopy(recording)
glm_job = prep.intensity_opticaldensity()
glm_job = prep.mbll(glm_job)
glm_job = prep.resample(glm_job)
glm_job = glm_module.GLM(glm_job)
glm_job.set_all_options({"Fs": 2, "noise_model": "ols", "verbose": False})
glm_recording = glm_job.run(glm_recording)
assert glm_recording["stats"] is not None
print("GLM fit completed with the recording quality mask applied.")

Detected 19213 bad time/channel samples in this recording
Resampled recording: 4207 masked samples, 4207 samples restored as NaN
PCA-corrected recording retains 19213 masked samples
GLM fit completed with the recording quality mask applied.


## Review the recordings yourself

Open the interactive viewer to inspect the recordings and their quality flags. Existing flags are loaded when the viewer opens. You can review the plots, try additional masking tools, and submit or cancel your edits.

The same viewer can also be opened from a regular Python script:

```python
from pyBrainAnalyzIR.vis.data_masking_manager import data_masking_manager

data_masking_manager(dset)
```

This example focuses on reviewing the data; submitting changes applies your chosen masks to the dataset.

In [4]:
from pyBrainAnalyzIR.vis.data_masking_manager import data_masking_manager

masking_window = data_masking_manager(dset)